# Quality checks

This notebook reproduces Figure 2 of Zada et al. (2025) from the results of `code/audioxcorr.py` and `code/erp.py`. We check in two ways that electrodes near early auditory cortex respond to the podcast. First, we cross-correlate each electrode's high-gamma activity with the audio envelope. Second, we average each electrode's activity around word onsets, also called an event-related potential (ERP).

Run `make audioxcorr erp` before this notebook.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from nilearn import datasets
from nilearn.plotting import plot_markers

from utils import load_results, get_coord_atlas_labels

plt.style.use("figures.mplstyle")
Path("../figures/figure2").mkdir(parents=True, exist_ok=True)

## Audio cross-correlation

We load the cross-correlations for every subject at once. Each electrode has one correlation per lag, from -10 to +10 seconds.

In [ ]:
xcorrs, coords, lags = load_results("audioxcorr")
print(f"Cross-correlations have a shape of: {xcorrs.shape}")

We summarize each electrode by its maximum correlation across lags and plot it on the brain. Sorting the values first puts the highest correlations on top.

In [ ]:
values = xcorrs.max(-1)
order = values.argsort()

fig, ax = plt.subplots(figsize=(2.2 * 3, 2), layout="constrained", dpi=300)
plot_markers(values[order], coords[order],
             node_size=10, display_mode="lzr",
             node_cmap="magma_r", colorbar=True,
             alpha=0.8, node_vmin=0,
             figure=fig, axes=ax)
fig.savefig("../figures/figure2/xcorr-brainmap.svg")
plt.show()

Next, we label every electrode with the Destrieux atlas so we can pull out the electrodes on lateral superior temporal gyrus (STG).

In [ ]:
atlas = datasets.fetch_atlas_destrieux_2009()
elec_labels = get_coord_atlas_labels(coords, atlas["maps"], atlas["labels"])
stg_mask = np.isin(elec_labels, ["G_temp_sup-Lateral"])
print(f"{stg_mask.sum()} electrodes are on lateral STG")

In [ ]:
fig = plot_markers(np.ones(stg_mask.sum()), coords[stg_mask],
                   node_size=15, display_mode="l",
                   alpha=0.8, colorbar=False,
                   node_cmap="Grays", node_vmin=0, node_vmax=1)
fig.savefig("../figures/figure2/stg-roi.svg")

Now we average the STG electrodes at each lag. Electrodes that follow the sound should peak shortly after lag 0.

In [ ]:
mean = xcorrs[stg_mask].mean(0)
err = xcorrs[stg_mask].std(0) / np.sqrt(stg_mask.sum())

fig, ax = plt.subplots(figsize=(2, 2))
ax.axvline(0, c="gray", ls=":")
ax.fill_between(lags, mean - err, mean + err, alpha=0.5, color="k")
ax.plot(lags, mean, color="k")
ax.set_xlabel("lag (s)")
ax.set_ylabel("r ± sem")
ax.text(0.01, 1, f"N={stg_mask.sum()}", fontsize=7, transform=ax.transAxes,
        verticalalignment="top", horizontalalignment="left")
ax.set_xlim(-2, 2)
fig.savefig("../figures/figure2/xcorr-timecourse.svg")
plt.show()

## Event-related potentials

The ERP results cover the same electrodes in the same order, so we can reuse the atlas labels from above.

In [ ]:
evoked, erp_coords, erp_lags = load_results("erp")
assert np.allclose(coords, erp_coords)
print(f"Evoked responses have a shape of: {evoked.shape}")

In [ ]:
values = evoked.max(-1)
order = values.argsort()

fig, ax = plt.subplots(figsize=(2.2 * 3, 2), layout="constrained", dpi=300)
plot_markers(values[order], coords[order],
             node_size=10, display_mode="lzr",
             node_cmap="magma_r", colorbar=True,
             alpha=0.8, node_vmin=0,
             figure=fig, axes=ax)
fig.savefig("../figures/figure2/erp-brainmap.svg")
plt.show()

And the average response of the STG electrodes around word onset:

In [ ]:
mean = evoked[stg_mask].mean(0)
err = evoked[stg_mask].std(0) / np.sqrt(stg_mask.sum())

fig, ax = plt.subplots(figsize=(2, 2))
ax.axvline(0, c="gray", ls=":")
ax.fill_between(erp_lags, mean - err, mean + err, alpha=0.5, color="k")
ax.plot(erp_lags, mean, color="k")
ax.set_xlabel("lag (s)")
ax.set_ylabel("zscore ± sem")
ax.text(0.01, 1, f"N={stg_mask.sum()}", fontsize=7, transform=ax.transAxes,
        verticalalignment="top", horizontalalignment="left")
ax.set_xlim(-2, 2)
fig.savefig("../figures/figure2/erp-timecourse.svg")
plt.show()